# Lesson 1.4 · Regularization and nonlinearity

Two ways to extend linear regression without changing model family: **penalizing** the coefficients to
reduce variance (Ridge, Lasso, ElasticNet), and **transforming the features** to capture curved relationships
(polynomials). Both require scaling the features, for reasons you will see in the cells.

**What you will learn**

- How Ridge and Lasso shrink the coefficients as α grows, and why only Lasso sets them to zero.
- To choose α with `RidgeCV` and `LassoCV`, inside a `Pipeline` with the scaler.
- To use `PolynomialFeatures` to fit a curve with a model that is linear in the parameters.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LassoCV, LinearRegression, Ridge, Lasso, RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

## Ridge and Lasso in two lines

A penalty on the size of the coefficients is added to the cost function: the sum of squares for
Ridge (L2), the sum of absolute values for Lasso (L1). The `alpha` parameter sets the strength: with α = 0 you are back
to OLS, with a very large α the coefficients are squashed toward zero.

```
Ridge:  minimizes  Σ (y − ŷ)² + α · Σ βⱼ²
Lasso:  minimizes  Σ (y − ŷ)² + α · Σ |βⱼ|
```

These are the textbook forms. scikit-learn writes the Lasso objective as 1/(2n) · Σ (y − ŷ)² + α · Σ |βⱼ|, dividing the squared error by 2n (Ridge does not divide), so the same `alpha`
means different things in the two models: an α chosen for the Lasso cannot be reused for the Ridge.

> **Common mistake.** The penalty hits the numeric value of the coefficients, which depends on the unit of measure of the
> feature. Without `StandardScaler` the regularization punishes the features based on how you measured them, not on how much they
> matter. The scaler goes **inside** the pipeline, so it is fitted only on the training fold.

In [ ]:
X, y = ds.diabetes()

ridge, lasso = {}, {}
for alpha in [0.01, 1, 10, 100, 1000]:
    ridge[alpha] = make_pipeline(StandardScaler(), Ridge(alpha=alpha)).fit(X, y)[-1].coef_
for alpha in [0.1, 1, 5, 10, 20]:
    lasso[alpha] = make_pipeline(StandardScaler(), Lasso(alpha=alpha)).fit(X, y)[-1].coef_

print("Ridge: the coefficients shrink but never reach zero")
display(pd.DataFrame(ridge, index=X.columns).round(1))
print("Lasso: some coefficients become exactly zero")
display(pd.DataFrame(lasso, index=X.columns).round(1))

## Choosing α

α is a hyperparameter: you don't choose it by looking at the training error (which always improves as α decreases) but with
cross-validation. `RidgeCV` and `LassoCV` do it in one go, trying a grid of values. The logarithmic
scale is the natural choice, because the effects of α show up by orders of magnitude.

In [ ]:
ridge_cv = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 25))).fit(X, y)
lasso_cv = make_pipeline(StandardScaler(), LassoCV(cv=5, random_state=0)).fit(X, y)

print("alpha chosen by RidgeCV:", round(ridge_cv[-1].alpha_, 3))
print("alpha chosen by LassoCV:", round(lasso_cv[-1].alpha_, 3))
print("features kept by the Lasso:", list(X.columns[lasso_cv[-1].coef_ != 0]))

## Nonlinearity with a linear model

"Linear" means linear **in the parameters**, not in the features. If you add x², x³, ... as columns the model
stays a linear regression, but the curve it draws in the (x, y) plane is no longer a line.
`PolynomialFeatures` does exactly this.

In [ ]:
d = ds.energy_consumption()                   # energy consumption against temperature: U-shaped relationship
X_t, y_t = d[["temperature"]], d["consumption"]

linear = LinearRegression().fit(X_t, y_t)
quadratic = make_pipeline(PolynomialFeatures(2, include_bias=False), LinearRegression()).fit(X_t, y_t)
print("linear R2   :", round(linear.score(X_t, y_t), 3))
print("quadratic R2:", round(quadratic.score(X_t, y_t), 3))

grid = pd.DataFrame({"temperature": np.linspace(-5, 35, 200)})
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.scatter(d.temperature, d.consumption, s=12, alpha=0.6)
ax.plot(grid, linear.predict(grid), label="degree 1", color="#9C4530")
ax.plot(grid, quadratic.predict(grid), label="degree 2", color="#1F6E5C")
ax.set(xlabel="temperature (C)", ylabel="consumption (kWh)"); ax.legend();

> **Warning.** High powers explode: with x between 0 and 30, x¹⁰ reaches 10¹⁴. With polynomials you scale **after**
> `PolynomialFeatures` (`make_pipeline(PolynomialFeatures(d), StandardScaler(), Ridge())`), and the degree must be treated
> as a hyperparameter to choose in cross-validation, like α.

## Transforming the target

When the target is skewed or the errors grow with the level, it is often better to model the logarithm.
`TransformedTargetRegressor` applies the transformation in fit and inverts it automatically in predict, so the
metrics stay in the original unit. It has a price that you will measure in exercise B1: the exponential of a mean of
logarithms is not the mean.

## Which one to choose

| Situation | Reasonable choice |
|---|---|
| Many features, almost all useful | Ridge |
| Many features, you suspect that few matter | Lasso |
| Groups of highly correlated features | ElasticNet |
| Curved relationship in one or two variables | PolynomialFeatures + Ridge, with the degree chosen in CV |
| Skewed target or errors proportional to the level | TransformedTargetRegressor with log |

## Check your understanding

> *Click on a question to show its answer.*

<details><summary>Why does the Lasso set coefficients to zero and Ridge doesn't?</summary>

Because of the geometry of the penalty. The allowed region for L1 is a diamond with its corners on the axes: the level curves of the error tend to touch it at a corner, where some coordinates are exactly zero. With L2 the region is a circle, with no corners, and the contact point is almost always generic.

</details>

<details><summary>Why does `LinearRegression` not need the scaler and `Ridge` does?</summary>

Because OLS is scale invariant: rescaling a feature rescales its coefficient in the opposite direction and the prediction does not change. The Ridge penalty, instead, depends on the value of the coefficients, so it depends on the units of measure. You will verify it in exercise F2.

</details>